In [26]:
import sys
from pathlib import Path
import importlib

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

wf = importlib.import_module("workflows.hpc")
# print(type(wf))

In [25]:
falcon_user  = ""
pheno_id     = "AD"
sumstats     = "dat/gwas/AD_ldsc_ready_neff.tsv"
n_cases      = 21982
n_controls   = 41944
genome_build = "GRCh37"
pqtl_dataset = "ukb_ppp"
pqtl_dir     = "dat/pqtls/ukb_ppp"
ref_bfile    = "ref/ldsc/1000G_EUR_Phase3_plink/1000G.EUR.QC.ALL"
postgres_db  = "drugmr"

In [ ]:
wf.clone_repo(falcon_user)
wf.container_checks(falcon_user)

In [ ]:
wf.run_gwas_qc(
    falcon_user=falcon_user,
    pheno_id=pheno_id,
    sumstats=sumstats,
    out_dir="results",
    snp_col="SNP",
    a1_col="A1",
    a2_col="A2",
    beta_col="BETA",
    se_col="SE",
    p_col="P",
    pos_col="BP",
    chr_col="CHR",
    af_col="FRQ",
    genome_build=genome_build,
    n_cases=n_cases,
    n_controls=n_controls,
)

In [ ]:
wf.run_cis_mr(
    falcon_user=falcon_user,
    pqtl_dataset=pqtl_dataset,
    pqtl_dir=pqtl_dir,
    pheno_id=pheno_id,
    pheno_gwas=f"results/QC/{pheno_id}/{pheno_id}.tsv",
    ref_bfile=ref_bfile,
)

In [ ]:
wf.load_postgres(
    falcon_user=falcon_user,
    pqtl_dataset=pqtl_dataset,
    pheno_id=pheno_id,
    db_id=postgres_db,
)

In [ ]:
wf.pull_results_local(
    falcon_user=falcon_user,
    pqtl_dataset=pqtl_dataset,
    pheno_id=pheno_id,
)

In [ ]:
wf.run_dashboard_local(
    db_name=postgres_db,
    phenotype=pheno_id,
    port_number=5432, # can be changed if needed
)